# Label
This notebook is used for labelling the bibtex data with the reference data, scope:
- Labelling manually 5 papers, each of these must ensure to have at least 20 rows in bibtex table
- Labelling automatically 500 papers in the remaining pool of bibtex table entries, which exclude manually labled rows.

The expected output of this notebook is a csv file containing ground truth of labelled data.

# Setup
This section is used to import necessary library.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from manual import ground_truth_manual

from rapidfuzz import fuzz

DIR_DATA = '../scopus/'
SAVE_DATA = '../dataset/'

# I. Exploration of scopus (bibtex and references table)

### Read data from scopus
- Read data of references table and bibtex table
- The bibtex table is splitted into shard for smaller size so it need to be concatenated when loaded again by pandas

In [2]:
files = sorted(Path(DIR_DATA + 'bibtex').glob("*.parquet"))

references_df = pd.read_parquet(DIR_DATA + 'references.parquet')
bibtex_df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

### Info lookup
- Explore the information of 2 tables

In [3]:
references_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81795 entries, 0 to 81794
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   arxiv_id             81795 non-null  str   
 1   ref_arxiv_id         81795 non-null  str   
 2   ref_title            81795 non-null  str   
 3   ref_authors          81795 non-null  object
 4   ref_submission_date  81651 non-null  str   
 5   ref_s2_id            81795 non-null  str   
dtypes: object(1), str(5)
memory usage: 14.8+ MB


In [4]:
bibtex_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 181344 entries, 0 to 181343
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   arxiv_id      181344 non-null  str     
 1   citation_key  181344 non-null  str     
 2   entry_type    181344 non-null  category
 3   title         177129 non-null  str     
 4   authors       181344 non-null  object  
 5   year          180171 non-null  float64 
 6   doi           71144 non-null   str     
 7   journal       149277 non-null  str     
 8   url           41541 non-null   str     
dtypes: category(1), float64(1), object(1), str(6)
memory usage: 36.0+ MB


### Row counting for each arXiv ID 
- With each arXiv ID, counting its number of rows in bibtex table

In [5]:
arXiv_row_count_bibtex = bibtex_df['arxiv_id'].value_counts()
arXiv_row_count_bibtex

arxiv_id
2608.07247    620
2608.07221    608
2608.05179    497
2608.06978    489
2608.05885    345
             ... 
2608.06884      2
2608.07737      2
2608.07842      2
2608.08251      2
2608.08458      1
Name: count, Length: 4274, dtype: int64

- With each arXiv ID, counting its number of rows in references table

In [6]:
arXiv_row_count_references = references_df['arxiv_id'].value_counts()
arXiv_row_count_references

arxiv_id
2608.07221    447
2608.07247    427
2608.05885    310
2608.07514    191
2608.05184    185
             ... 
2608.08845      1
2608.08933      1
2608.08934      1
2608.08993      1
2608.08997      1
Name: count, Length: 4288, dtype: int64

### Filter rows that are appropriate with requirements
- We only those arXiv IDs which have the number of rows is equal or greater than 20
- I gonna choose the upper bound as 30 for easily manual label :))

In [7]:
arXiv_filter_bibtex = arXiv_row_count_bibtex[(arXiv_row_count_bibtex >= 20) & (arXiv_row_count_bibtex <= 30)].index
len(arXiv_filter_bibtex)

869

- We also do the same thing with references table
- This is for effective and sufficient labelling later

In [8]:
arXiv_filter_references = arXiv_row_count_references[(arXiv_row_count_references >= 20) & (arXiv_row_count_references <= 30)].index
len(arXiv_filter_references)

810

- Besides the appropriate row quantity, the arXiv IDs must also be existent in references table

In [9]:
arXiv_filter = arXiv_filter_bibtex[arXiv_filter_bibtex.isin(arXiv_filter_references)]
len(arXiv_filter)

97

- Oh, there are only 97 appropriate arXiv IDs for manual labelling.

In [10]:
bibtex_filter_auto = arXiv_row_count_bibtex[arXiv_row_count_bibtex >= 2].index
ref_filter_auto = arXiv_row_count_references[arXiv_row_count_references >= 2].index
arXiv_filter_auto = bibtex_filter_auto[bibtex_filter_auto.isin(ref_filter_auto)]
len(arXiv_filter_auto)

3555

# II. Manual Labelling

### Choosing samples
- I randomly choose 5 five samples from filter arXiv ID set

In [11]:
rng = np.random.default_rng(42)
selected = rng.choice(arXiv_filter, size=5, replace=False)
selected

array(['2608.06628', '2608.06434', '2608.04791', '2608.06791',
       '2608.05850'], dtype=object)

- I have chosen 5 arXiv IDs from the filter set (this ensures the selected publications to have at least 20 rows in bibtex table)
- Let see the number of rows for each of these arXiv ID in bibtex table.

In [12]:
arXiv_row_count_bibtex[selected]

arxiv_id
2608.06628    25
2608.06434    28
2608.04791    26
2608.06791    30
2608.05850    20
Name: count, dtype: int64

- Firstly, we must ensure those selected arXiv IDs to exist in references dataframe (these are filtered above, but we should perform this again to check)

In [13]:
mask_sum = np.isin(selected,arXiv_row_count_references.index).sum()
mask_sum == len(selected)

np.True_

- Good, all arXiv IDs that we have selected exist in references table
- Let see the number of rows corresponding with these arXiv IDs in references table

In [14]:
arXiv_row_count_references[selected]

arxiv_id
2608.06628    22
2608.06434    27
2608.04791    22
2608.06791    22
2608.05850    21
Name: count, dtype: int64

- Let see the first publication for manual labelling this

In [15]:
bibtex_rows = bibtex_df.loc[bibtex_df['arxiv_id'] == selected[4]]
ref_rows = references_df.loc[references_df['arxiv_id'] == selected[4]]

In [16]:
bibtex_rows

,arxiv_id,citation_key,entry_type,title,authors,year,doi,journal,url
68819,2608.05850,nllb-24,article,Scaling neural machine translation to 200 lang...,"[NLLB Team, Marta R. Costa-jussà, James Cross,...",2024.0,10.1038/s41586-024-07335-x,Nature,https://doi.org/10.1038/s41586-024-07335-x
68820,2608.05850,sentalign-2023,inproceedings,SentAlign: Accurate and Scalable Sentence Alig...,"[Steinþór Steingrímsson, Hrafn Loftsson, Andy ...",2023.0,NaN,Proceedings of the 2023 Conference on Empirica...,NaN
68821,2608.05850,wu2025bitterlessonlearned2000,misc,"The Bitter Lesson Learned from 2,000+ Multilin...","[Minghao Wu, Weixuan Wang, Sinuo Liu, Huifeng ...",2025.0,NaN,NaN,https://arxiv.org/abs/2504.15521
68822,2608.05850,urieli2025jochre,article,Jochre 3 and the Yiddish OCR corpus,"[Assaf Urieli, Amber Clooney, Michelle Sigiel,...",2025.0,NaN,arXiv preprint arXiv:2501.08442,NaN
68823,2608.05850,yiddishUD,misc,YiTB: the Yiddish Tree Bank,[Matthew Kirk Andrews],2025.0,NaN,NaN,https://github.com/UniversalDependencies/UD_Yi...
68824,2608.05850,berkovitch_rusinek_yiddish_2022,misc,New Languages for NLP: Yiddish — Annotated Dat...,"[Ephraim Berkovitch, Sinai Rusinek]",2021.0,NaN,NaN,https://github.com/New-Languages-for-NLP/yiddish
68825,2608.05850,koto2025sherkalachat,inproceedings,Sherkala-Chat: Building a State-of-the-Art LLM...,"[Fajri Koto, Rituraj Joshi, Nurdaulet Mukhitul...",2025.0,NaN,Second Conference on Language Modeling,https://openreview.net/forum?id=wRcTCcb0H5
68826,2608.05850,bleaman2025corpus,article,"The corpus of spoken Yiddish in Europe: Goals,...","[Isaac L Bleaman, Chaya R Nove]",2025.0,NaN,Language Documentation & Conservation,NaN
68827,2608.05850,webber2022reyd,inproceedings,REYD-The First Yiddish Text-to-Speech Dataset ...,"[Jacob Webber, Samuel K Lo, Isaac L Bleaman]",2022.0,NaN,INTERSPEECH,NaN
68828,2608.05850,kulick2022part,article,A part-of-speech tagger for Yiddish,"[Seth Kulick, Neville Ryant, Beatrice Santorin...",2022.0,NaN,arXiv preprint arXiv:2204.01175,NaN


In [17]:
ref_rows

,arxiv_id,ref_arxiv_id,ref_title,ref_authors,ref_submission_date,ref_s2_id
30648,2608.05850,2506.00288,Emergent Abilities of Large Language Models un...,"[Ahmed Elhady, Eneko Agirre, Mikel Artetxe]",2025-05-30,e943d1ac1e74d4290a7bf8e16d01394d964220ef
30649,2608.05850,2505.22232,Judging Quality Across Languages: A Multilingu...,"[Mehdi Ali, Manuel Brack, Max Lübbering, Elias...",2025-05-28,5df67582cd7ed394c0180ca2e3ab59ce7c0516a8
30650,2608.05850,2505.09388,Qwen3 Technical Report,"[An Yang, An-Feng Li, Bao-Song Yang, Bei-Chen ...",2025-05-14,d2d84d56f730f81d276a02b48d5d44db5bde0b4a
30651,2608.05850,2504.15521,"The Bitter Lesson Learned from 2,000+ Multilin...","[Minghao Wu, Wei-Xuan Wang, Sinuo Liu, Huifeng...",2025-04-22,4809f38b9162ae3609a44129c8fef6d8a8d9ecaf
30652,2608.05850,2503.01493,Sherkala-Chat: Building a State-of-the-Art LLM...,"[Fajri Koto, Rituraj Joshi, Nurdaulet Mukhitul...",2025-03-03,2ddcbb6189731a8e4e088dff15ae3f333b7752e6
30653,2608.05850,2501.08442,Jochre 3 and the Yiddish OCR corpus,"[A. Urieli, Amber Clooney, Michelle Sigiel, Gr...",2025-01-14,89ef57c269ea3977862be9bc34c6a2ad940a45c2
30654,2608.05850,2408.02239,Pula: Training Large Language Models for Setswana,"[Nathan Brown, V. Marivate]",2024-08-05,0dc574a4d9788285a8d051bf0a7104c7584bb3ec
30655,2608.05850,2407.21783,The Llama 3 Herd of Models,"[Aaron Grattafiori, Abhimanyu Dubey, Abhinav J...",2024-07-31,9da8ea6466d7142703d8771d8dcea44e19304abb
30656,2608.05850,2404.04042,Teaching Llama a New Language Through Cross-Li...,"[Hele-Andra Kuulmets, Taido Purason, Agnes Luh...",2024-04-05,8dc4ac4446f787f0647e1e16a9c43b2880e04008
30657,2608.05850,2403.20266,Latxa: An Open Language Model and Evaluation S...,"[Julen Etxaniz, Oscar Sainz, Naiara Pérez, Itz...",2024-03-29,7cccc78f24e4eeca5356151b3d3557c1f45b6294


- Manual labelling with 5 publications

In [18]:
ground_truth_manual.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   arxiv_id      102 non-null    str  
 1   citation_key  102 non-null    str  
 2   ref_arxiv_id  102 non-null    str  
dtypes: str(3)
memory usage: 5.9 KB


In [19]:
ground_truth_manual.head()

,arxiv_id,citation_key,ref_arxiv_id
0,2608.06628,kim2025train,2502.06768
1,2608.06628,ye2024beyond,2410.14157
2,2608.06628,song2025seed,2508.02193
3,2608.06628,trainin2026discrete,2602.19980
4,2608.06628,schiff2025simple,2412.10193


- Let reorganize the ground truth

In [20]:
truth = ground_truth_manual.rename(
    columns={"ref_arxiv_id": "true_arxiv_id"}
)

candidates = (
    references_df[["arxiv_id", "ref_arxiv_id"]]
    .rename(columns={"ref_arxiv_id": "candidate_arxiv_id"})
)

manual_labeled = truth.merge(candidates, on="arxiv_id", how="inner")

manual_labeled["label"] = (
    manual_labeled["candidate_arxiv_id"] == manual_labeled["true_arxiv_id"]
).astype(int)

manual_labeled = manual_labeled[["arxiv_id", "citation_key", "candidate_arxiv_id", "label"]]

manual_labeled.info()

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   arxiv_id            2370 non-null   str  
 1   citation_key        2370 non-null   str  
 2   candidate_arxiv_id  2370 non-null   str  
 3   label               2370 non-null   int64
dtypes: int64(1), str(3)
memory usage: 152.8 KB


- Let save the manual ground truth into a csv file

In [21]:
manual_labeled.to_csv(SAVE_DATA + 'manual_truth.csv', index=False)

## III. Automatical Labelling

- Firstly, I gonna choose 500 publications in the remaining pool, excluding those are manual labelled
- There is no requirement for the number of rows for these automatical publications

In [22]:
remaining_pool = arXiv_filter_auto[~arXiv_filter_auto.isin(selected)]

rng = np.random.default_rng(42)
selected_auto = rng.choice(remaining_pool, size=500, replace=False)
selected_auto[:5]

array(['2608.06981', '2608.07137', '2608.05829', '2608.06582',
       '2608.06575'], dtype=object)

- The algorithm for automatical labelling is fuzz score
- It states that the score reflects the fit rate between two entries. The higher the score is, the higher the fit rate is 

In [23]:
MARGIN = 10
THRESHOLD = 95

auto_truth = []
for paper_id in selected_auto:
    bib_rows = bibtex_df.loc[bibtex_df['arxiv_id'] == paper_id]
    ref_rows= references_df.loc[references_df['arxiv_id'] == paper_id]

    for _, bib_entry in bib_rows.iterrows():
        scores = []
        if not isinstance(bib_entry['title'], str) or not bib_entry['title'].strip():
            continue 
        for _, candidate in ref_rows.iterrows():
            score = fuzz.ratio(
                bib_entry['title'].lower().strip(),
                candidate['ref_title'].lower().strip()
            )
            scores.append(score)
        scores = np.array(scores)
        second_idx, best_idx = np.argsort(scores)[-2:]
        best_score, second_score = scores[best_idx], scores[second_idx]
        accept = (
            (best_score >= THRESHOLD)
            and (best_score - second_score >= MARGIN)
        )
        print (f"Paper ID: {paper_id}, Bib Entry: {bib_entry['citation_key']}, Best Score: {best_score}, Second Score: {second_score}, Accept: {accept}")
        print(f'Best candidate: {ref_rows.iloc[best_idx]["ref_title"]}, Second candidate: {ref_rows.iloc[second_idx]["ref_title"]}')
        print(f'Bibtex title: {bib_entry["title"]}')
        print()
        if accept:
            auto_truth.append({
                'arxiv_id': bib_entry['arxiv_id'],
                'citation_key': bib_entry['citation_key'],
                'ref_arxiv_id': ref_rows.iloc[best_idx]['ref_arxiv_id'],
            })

Paper ID: 2608.06981, Bib Entry: DBLP:conf/icml/Sohl-DicksteinW15, Best Score: 100.0, Second Score: 47.42268041237113, Accept: True
Best candidate: Deep Unsupervised Learning using Nonequilibrium Thermodynamics, Second candidate: Denoising Diffusion Implicit Models
Bibtex title: Deep Unsupervised Learning using Nonequilibrium Thermodynamics

Paper ID: 2608.06981, Bib Entry: DBLP:conf/nips/HoJA20, Best Score: 82.66666666666667, Second Score: 54.90196078431373, Accept: False
Best candidate: Denoising Diffusion Implicit Models, Second candidate: Denoising Diffusion Models for Plug-and-Play Image Restoration
Bibtex title: Denoising Diffusion Probabilistic Models

Paper ID: 2608.06981, Bib Entry: DBLP:conf/iclr/0011SKKEP21, Best Score: 100.0, Second Score: 55.319148936170215, Accept: True
Best candidate: Score-Based Generative Modeling through Stochastic Differential Equations, Second candidate: Generative Modeling by Estimating Gradients of the Data Distribution
Bibtex title: Score-Based G

In [24]:
ground_truth_auto = pd.DataFrame(auto_truth)
ground_truth_auto.info()
assert set(ground_truth_auto["arxiv_id"]).isdisjoint(selected)

<class 'pandas.DataFrame'>
RangeIndex: 8625 entries, 0 to 8624
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   arxiv_id      8625 non-null   str  
 1   citation_key  8625 non-null   str  
 2   ref_arxiv_id  8625 non-null   str  
dtypes: str(3)
memory usage: 498.8 KB


In [25]:
ground_truth_auto.head()

,arxiv_id,citation_key,ref_arxiv_id
0,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,1503.03585
1,2608.06981,DBLP:conf/iclr/0011SKKEP21,2011.13456
2,2608.06981,DBLP:conf/nips/SongE19,1907.05600
3,2608.06981,DBLP:conf/cvpr/ZhuZLCWTG23,2305.08995
4,2608.06981,DBLP:conf/iclr/WangYZ23,2212.00490


- Check the automatical labelling set whether it has duplicate rows (taking arxiv_id and ref_arxiv_id column into consideration).

In [26]:
ground_truth_auto. duplicated(subset=['arxiv_id', 'ref_arxiv_id']).sum()

mask = ground_truth_auto.duplicated(subset=['arxiv_id', 'ref_arxiv_id'], keep=False)
ground_truth_auto.loc[mask]

,arxiv_id,citation_key,ref_arxiv_id
2398,2608.07778,leung_nomoto_2018,2007.08466
2401,2608.07778,leung_nomoto_2020a,2007.08466
2402,2608.07778,leung_nomoto_2020b,2007.08466
2485,2608.04353,2025SCPMA..6800413H,2502.04212
2489,2608.04353,2014A&A...571A..16P,1303.5076
2498,2608.04353,2024arXiv240403000D,2404.03000
2499,2608.04353,2024arXiv240403002D,2404.03002
2506,2608.04353,planck2018,1303.5076
2510,2608.04353,2025JCAP...04..012A,2404.03000
2512,2608.04353,desidr1,2404.03002


- Oh, there are some duplicated rows here
- I gonna check them whether they have the same title

In [27]:
mask_df = ground_truth_auto[mask][['arxiv_id', 'citation_key']]

matched_bibtex = bibtex_df.merge(
    mask_df,
    on=["arxiv_id", "citation_key"],
    how="inner"
)

matched_bibtex

,arxiv_id,citation_key,entry_type,title,authors,year,doi,journal,url
0,2608.04353,2025SCPMA..6800413H,article,The DESI DR1/DR2 evidence for dynamical dark e...,"[Lu Huang, Rong-Gen Cai, Shao-Jiang Wang]",2025.0,10.1007/s11433-025-2754-5,Sci. China Phys. Mech. Astron.,NaN
1,2608.04353,2014A&A...571A..16P,article,Planck 2013 results. XVI. Cosmological parameters,"[Planck Collaboration, P. A. R. Ade, N. Aghani...",2014.0,10.1051/0004-6361/201321591,NaN,NaN
2,2608.04353,2024arXiv240403000D,article,DESI 2024 III: Baryon Acoustic Oscillations fr...,"[DESI Collaboration, A. G. Adame, J. Aguilar, ...",2024.0,10.48550/arxiv.2404.03000,arXiv e-prints,NaN
3,2608.04353,2024arXiv240403002D,article,DESI 2024 VI: Cosmological Constraints from th...,"[DESI Collaboration, A. G. Adame, J. Aguilar, ...",2024.0,10.48550/arxiv.2404.03002,arXiv e-prints,NaN
4,2608.04353,planck2018,article,Planck 2018 results. VI. Cosmological parameters,"[Planck Collaboration, N. Aghanim, Y. Akrami, ...",2020.0,10.1051/0004-6361/201833910,NaN,NaN
5,2608.04353,2025JCAP...04..012A,article,DESI 2024 III: baryon acoustic oscillations fr...,"[A. G. Adame, J. Aguilar, S. Ahlen, S. Alam, D...",2025.0,10.1088/1475-7516/2025/04/012,NaN,NaN
6,2608.04353,desidr1,article,DESI 2024 VI: cosmological constraints from th...,"[A. G. Adame, J. Aguilar, S. Ahlen, S. Alam, D...",2025.0,10.1088/1475-7516/2025/02/021,NaN,NaN
7,2608.04353,2025arXiv250204212H,article,The DESI DR1/DR2 evidence for dynamical dark e...,"[Lu Huang, Rong-Gen Cai, Shao-Jiang Wang]",2025.0,10.48550/arxiv.2502.04212,arXiv e-prints,NaN
8,2608.05068,saparov2022lattice,article,Lattice dynamics with molecular Berry curvatur...,"[Daniyar Saparov, Bangguo Xiong, Yafei Ren, Qi...",2022.0,NaN,Physical Review B,NaN
9,2608.05068,zhang2022lattice,article,Lattice dynamics with molecular Berry curvatur...,"[Lishu Zhang, Qian Niu]",2022.0,NaN,Physical Review B,NaN


- These duplicated rows have the same title in their own publication, so I gonna keep them in the ground truth
- I gonna do the same thing like the manual label stage: reorganize the ground truth dataframe and then save it into a csv file

In [28]:
truth_auto = ground_truth_auto.rename(
    columns={"ref_arxiv_id": "true_arxiv_id"}
)

candidates = (
    references_df[["arxiv_id", "ref_arxiv_id"]]
    .rename(columns={"ref_arxiv_id": "candidate_arxiv_id"})
)

auto_labeled = truth_auto.merge(candidates, on="arxiv_id", how="inner")

auto_labeled["label"] = (
    auto_labeled["candidate_arxiv_id"] == auto_labeled["true_arxiv_id"]
).astype(int)

auto_labeled = auto_labeled[["arxiv_id", "citation_key", "candidate_arxiv_id", "label"]]

auto_labeled.info()

<class 'pandas.DataFrame'>
RangeIndex: 263479 entries, 0 to 263478
Data columns (total 4 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   arxiv_id            263479 non-null  str  
 1   citation_key        263479 non-null  str  
 2   candidate_arxiv_id  263479 non-null  str  
 3   label               263479 non-null  int64
dtypes: int64(1), str(3)
memory usage: 17.0 MB


In [29]:
auto_labeled.to_csv(SAVE_DATA + 'auto_truth.csv', index=False)

We have finished the label stage here !!